# 26 — Invoice Extraction
> Plain invoice text goes in. Vendor, dates, line items, and totals come out — fully structured and ready to load into any system.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q pydantic-ai openai pydantic
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
import os
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel


# --- Data shapes ---

class LineItem(BaseModel):
    description: str = Field(description="Description of the goods or service.")
    quantity: float = Field(description="Number of units.")
    unit_price: float = Field(description="Price per unit in the invoice currency.")
    total: float = Field(description="Line total (quantity x unit_price).")


class Invoice(BaseModel):
    vendor: str = Field(description="Name of the company or individual issuing the invoice.")
    invoice_number: str = Field(description="Invoice reference number or ID.")
    invoice_date: str = Field(description="Date the invoice was issued (YYYY-MM-DD if parseable, else raw string).")
    due_date: str = Field(description="Payment due date (YYYY-MM-DD if parseable, else raw string).")
    currency: str = Field(description="Three-letter ISO currency code (e.g. USD, EUR, GBP).")
    subtotal: float = Field(description="Sum of all line item totals before tax.")
    tax: float = Field(description="Total tax amount. Use 0.0 if not stated.")
    total_due: float = Field(description="Total amount due including tax.")
    line_items: list[LineItem] = Field(description="List of individual line items on the invoice.")


# --- Extraction logic ---

def extract(invoice_text: str, model_name: str = "gpt-4o-mini") -> Invoice:
    """Extract structured invoice data from raw invoice text."""
    model = OpenAIModel(model_name, api_key=os.environ["OPENAI_API_KEY"])
    agent = Agent(
        model,
        result_type=Invoice,
        system_prompt=(
            "You are an invoice parsing assistant. "
            "Extract all fields exactly as stated in the document. "
            "If a field is missing, use an empty string for text fields and 0.0 for numeric fields."
        ),
    )
    result = agent.run_sync(f"Extract the invoice fields from the following:\n\n{invoice_text}")
    return result.data

## The scenario

A construction materials supplier has sent an invoice with four line items: two product deliveries, a delivery surcharge, and a handling fee. The document arrives as plain text with mixed units, non-standard date formatting, and a 10% GST charge buried at the bottom. The agent reads the document and returns every field structured and ready to post directly to accounts payable.

In [ ]:
INVOICE_TEXT = """
TAX INVOICE

Supplier: Hartwell Building Supplies Pty Ltd
Invoice No: HBS-2025-1147
Invoice Date: 12 March 2025
Payment Due: 11 April 2025
Currency: AUD

Bill To: Meridian Construction Group

Items Supplied:
  Structural steel beams (Grade 350)   8 units    @ AUD 1,240.00 each    AUD 9,920.00
  Concrete reinforcing mesh (sheets)   120 sheets @ AUD 38.50 each       AUD 4,620.00
  Site delivery -- crane offload        1 service  @ AUD 875.00            AUD 875.00
  Materials handling & staging fee     1 service  @ AUD 450.00            AUD 450.00

Subtotal:      AUD 15,865.00
GST (10%):     AUD 1,586.50
TOTAL DUE:     AUD 17,451.50

Payment terms: Net 30 days. EFT preferred.
BSB: 062-000  Account: 1234 5678
"""

invoice = extract(INVOICE_TEXT)

print(f"VENDOR:       {invoice.vendor}")
print(f"INVOICE #:    {invoice.invoice_number}")
print(f"ISSUED:       {invoice.invoice_date}")
print(f"DUE:          {invoice.due_date}")
print(f"CURRENCY:     {invoice.currency}")
print()
print(f"{'DESCRIPTION':<40} {'QTY':>6}  {'UNIT PRICE':>12}  {'TOTAL':>12}")
print("-" * 74)
for item in invoice.line_items:
    print(f"{item.description:<40} {item.quantity:>6.1f}  {item.unit_price:>12,.2f}  {item.total:>12,.2f}")
print("-" * 74)
print(f"{'Subtotal':<40} {'':>6}  {'':>12}  {invoice.subtotal:>12,.2f}")
print(f"{'Tax':<40} {'':>6}  {'':>12}  {invoice.tax:>12,.2f}")
print(f"{'TOTAL DUE':<40} {'':>6}  {'':>12}  {invoice.total_due:>12,.2f}")

## Use your own data

Replace the input above with:
- `INVOICE_TEXT` — paste in any invoice as plain text (PDFs can be converted with a tool like `pdfplumber` or copied manually)
- The currency, tax rate, and number of line items do not need to match — the agent adapts to whatever the document contains

The agent will return vendor details, all dates, every line item with quantity and unit price, and the final total — structured and ready to insert into your database or ERP.